In [1]:
import os
import sys

PROJECT_PATH = os.getcwd()
if not os.path.isdir(os.path.join(PROJECT_PATH, "srcs")):
    PROJECT_PATH = os.path.dirname(PROJECT_PATH)

if PROJECT_PATH not in sys.path:
    sys.path.insert(0, PROJECT_PATH)

In [2]:
from argparse import Namespace

from scripts.train import Trainer, load_dataset
from srcs.datasets.vicocktail_dataset import DataCollator, VideoTransform
from srcs.nets.pytorch_backend.e2e_vsr_conformer import E2ECommon
from srcs.nlp.norm import *
from srcs.nlp.text_transform import TextTransform
from srcs.nlp.tokenizer import WordTokenizer

d:\projects\VietnameseVSR\venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [8]:
TEST_OUTPUT_DIR = os.path.join(PROJECT_PATH, "checkpoints", "train_10_percent")
TEST_VOCAB_PATH = os.path.join(TEST_OUTPUT_DIR, "vocab.txt")

args = Namespace(
    model="e2ecommon",
    dataset="vicocktail",
    output_dir=TEST_OUTPUT_DIR,
    train_fraction=0.1,
    val_fraction=1.0,
    epochs=1,
)
args

Namespace(model='e2ecommon', dataset='vicocktail', output_dir='d:\\projects\\VietnameseVSR\\checkpoints\\train_10_percent', train_fraction=0.1, val_fraction=1.0, epochs=1)

In [4]:
dataset = load_dataset(args)

print(dataset)
print(f"Train samples: {len(dataset['train']):,}")
print(f"Validation samples: {len(dataset['val']):,}")

DatasetDict({
    train: Dataset({
        features: ['label', 'length', 'sample_id', 'video'],
        num_rows: 18823
    })
    val: Dataset({
        features: ['label', 'length', 'sample_id', 'video'],
        num_rows: 5844
    })
})
Train samples: 18,823
Validation samples: 5,844


In [5]:
def iter_labels(dataset_split):
    for label in dataset_split["label"]:
        if isinstance(label, (bytes, bytearray, memoryview)):
            label = bytes(label).decode("utf-8")
        yield str(label)


normalizer = TextNormalizer(
    lowercase=True,
    rules=[
        RemovePunctNormalizer(),
        RemoveNumericNormalizer(),
        RemoveSpecialCharNormalizer(),
        SpaceNormalizer(),
    ],
)
text_transform = TextTransform(
    tokenizer=WordTokenizer(normalizer),
    vocab_path=TEST_VOCAB_PATH,
)
text_transform.create_vocab(iter_labels(dataset["train"]), min_frequency=1)

print(f"Vocabulary size: {len(text_transform.token_list):,}")

Vocabulary size: 3,868


In [10]:
if args.model != "e2ecommon":
    raise ValueError(f"Unsupported model: {args.model}")

model = E2ECommon(
    odim=len(text_transform.token_list),
    num_blocks=4,
)
train_collator = DataCollator(
    text_transform=text_transform,
    video_transform=VideoTransform(subset="train"),
)
val_collator = DataCollator(
    text_transform=text_transform,
    video_transform=VideoTransform(subset="val"),
)
trainer = Trainer(
    datasets=dataset,
    model=model,
    train_collator=train_collator,
    val_collator=val_collator,
    output_dir=args.output_dir,
    epochs=args.epochs,
    batch_size=2,
    num_workers=0,
)

print(f"Model: {model.__class__.__name__}")
print(f"Conformer blocks: {len(model.encoder.encoders)}")
print(f"Device: {trainer.device}")

Model: E2ECommon
Conformer blocks: 4
Device: cuda


In [9]:
trainer.run()

Device: cuda
Train samples: 18,823
Validation samples: 5,844


Epoch 1/100 [train]:   7%|▋         | 686/9412 [02:37<33:26,  4.35it/s, avg_loss=6.5807, loss=6.1401, lr=1.00e-04]


KeyboardInterrupt: 

In [ ]:
for checkpoint_name in ("last.pt", "best.pt"):
    checkpoint_path = os.path.join(TEST_OUTPUT_DIR, checkpoint_name)
    print(
        checkpoint_name,
        "exists:",
        os.path.isfile(checkpoint_path),
    )